In [1]:
import sys
sys.path.append(r"C:\Users\hjia9\Documents\GitHub\data-analysis")
sys.path.append(r"C:\Users\hjia9\Documents\GitHub\data-analysis\ucla-lapd")
import os
import numpy as np
import matplotlib.pyplot as plt
import copy
from scipy.ndimage import gaussian_filter1d, gaussian_filter
from scipy.signal import find_peaks, savgol_filter

import read_hdf5 as rh
from bapsflib import lapd

from lp_analysis import find_sweep_indices, reshape_IV
from lp_iv_analysis import analyze_IV


%matplotlib qt
plt.rcParams.update({'font.size': 16})

In [2]:
ifn = r"D:\data\LAPD\Mar26-data\11-lp-sweep-xyplane-bias.hdf5"

# Extract directory and run number from ifn
data_dir = os.path.dirname(ifn)
run_num = os.path.basename(ifn).split('-')[0]  # "11" from "11-lp-sweep-xyplane-bias.hdf5"
# Create output filename
save_path = os.path.join(data_dir, f"{run_num}-sweep-data.npz")

with lapd.File(ifn) as f:
    rh.show_info(f)
    adc, digi_dict = rh.read_digitizer_config(f)
    pos_array, xpos, ypos, zpos, npos, nshot = rh.read_bmotion_probe_motion(f)
    
data = np.load(save_path)

11-lp-sweep-xyplane-bias.hdf5 Overview
Generated by bapsflib (v0.0.0)
Generated date: 3/17/2026 7:53:49 PM


Filename:     11-lp-sweep-xyplane-bias.hdf5
Abs. Path:    D:\data\LAPD\Mar26-data\11-lp-sweep-xyplane-bias.hdf5
LaPD version: 1.2
Investigator: Jia Han
Run Date:     3/11/2026 9:01:59 PM

Exp. and Run Structure:
  (set)  Electrode_Biasing
  (exp)  +-- mar2026-jia
  (run)  |   +-- 11-lp-sweep-xyplane-bias

Run Description:
    Recording langmuir probe sweep signal with bias; same condition as run09.
    Most diagnostic and bias settings same as feb2026 experiments.
    
    
    LAPD B field:
    Black magnets at south: 888 A (PS12, 13),
    Yellow & Purple magnets: configured for uniform 800 G
    Except Yellow magnet PS 3, 4: 0 kG
    Black magnets at north: 0 A (PS11)
    (1600G-Source: 800 G: Bulk, 0 G: north end)
    
    South LaB6 source:
    He plasma, Discharge PS Voltage: 80 V, discharge current: 4.3 kA
    65 V cathode-anode voltage,   1/3 Hz rep rate 
    Heater: 34.6

In [3]:
t_ls = data['data_timestamp']

In [4]:
ps_path = r"D:\data\LAPD\Mar26-data\11-plasma-data.npz"
ps_data = np.load(ps_path)
Vp_arr = ps_data['Vp_arr']
Te_arr = ps_data['Te_arr']
ne_arr = ps_data['ne_arr']
Vp_err = ps_data['Vp_err']
Te_err = ps_data['Te_err']
ne_err = ps_data['ne_err']

In [6]:
extent = min(xpos), max(xpos), min(ypos), max(ypos)

sweep_idx = 10
grid_shape = (61, 61) 
# Extract the data for the requested sweep and reshape to 2D
Vp_2d = Vp_arr[:, sweep_idx].reshape(grid_shape)
Te_2d = Te_arr[:, sweep_idx].reshape(grid_shape)
ne_2d = ne_arr[:, sweep_idx].reshape(grid_shape)

# Create a base colormap and tell it to render NaNs as white
cmap = copy.copy(plt.cm.viridis)
cmap.set_bad(color='white')


# Set up a 1x3 panel figure
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
t_wt_bias = t_ls[sweep_idx]*1e3 + 14 - 15.5
fig.suptitle(f'2D Spatial Plasma Profile (t = {t_wt_bias:.2f} ms)', fontsize=32, fontweight='bold')

# --- 1. Plasma Potential (Vp) ---
im0 = axs[0].imshow(Vp_2d, origin='lower', cmap=cmap, extent=extent, vmin=0, vmax=30)
axs[0].set_title('Plasma Potential ($V_p$) [V]')
axs[0].set_xlabel('X Position')
axs[0].set_ylabel('Y Position')
fig.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)

# --- 2. Electron Temperature (Te) ---
im1 = axs[1].imshow(Te_2d, origin='lower', cmap=cmap, extent=extent, vmin=0, vmax=3)
axs[1].set_title('Electron Temp ($T_e$) [eV]')
axs[1].set_xlabel('X Position')
# Y-label hidden for the middle plot to save space
fig.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

# --- 3. Electron Density (ne) ---
im2 = axs[2].imshow(ne_2d, origin='lower', cmap=cmap, extent=extent, interpolation='gaussian', vmin=1e11, vmax=1e13)
axs[2].set_title('Electron Density ($n_e$) [cm$^{-3}$]')
axs[2].set_xlabel('X Position')
fig.colorbar(im2, ax=axs[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [8]:
ind = 68
tnd = 10
voltage = data['Vswp_arr_rs'][ind,tnd]
current = data['IswpL_arr_rs'][ind,0,tnd]

plt.figure(figsize=(12, 8))
plt.plot(voltage, current)
plt.plot(voltage, gaussian_filter1d(current, 10))

current = gaussian_filter1d(current, 10)

In [9]:
Vp, Te, ne = analyze_IV(voltage, current, plot=True)
print(f"Vp = {Vp:.2f} V, Te = {Te:.2f} eV, ne = {ne:.2e} m^-3")

Vp = 17.44 V, Te = 99999.05 eV, ne = 2.14e+09 m^-3


In [ ]:
with lapd.File(ifn) as f:
    data, tarr = rh.read_data(f, 4, 5, index_arr=slice(npos*nshot), adc=adc)
    Vswp_arr = data['signal'].reshape((npos, nshot, -1)) * 100 # X50 probe
    Vswp_arr = np.mean(Vswp_arr, axis=1)

    data, tarr = rh.read_data(f, 4, 4, index_arr=slice(npos*nshot), adc=adc)
    Iswp_arr = data['signal'].reshape((npos, nshot, -1)) / (7.2*2e-3) # 7.2 Ohm with 2mm^2 area